In [1]:
import pandas as pd
import numpy as np
import os
building_csv = pd.read_csv("/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/격자별 평균 고도 + 건물 높이 합, 건물 면적 합/grid_with_building_stats.csv")

In [2]:
building_csv.head()

,Unnamed: 0,CELL_ID,CELL_X,CELL_Y,GID,lon,lat,elev_mean,geometry,sum_area,sum_height
0,0,다사54504050,954625,1940625,다사54ba40ba,126.985479,37.464854,67.960381,MULTIPOLYGON (((198716.7246971111 540351.97920...,0.0,0.0
1,1,다사54255675,954375,1956875,다사54ab56bb,126.981639,37.611306,236.999630,MULTIPOLYGON (((198380.2035844773 556606.50577...,0.0,0.0
2,2,다사54255700,954375,1957125,다사54ab57aa,126.981624,37.613560,239.770579,MULTIPOLYGON (((198378.87163139007 556856.5958...,0.0,0.0
3,3,다사54255725,954375,1957375,다사54ab57ab,126.981608,37.615813,285.148324,MULTIPOLYGON (((198377.53961030743 557106.6858...,0.0,0.0
4,4,다사54255750,954375,1957625,다사54ab57ba,126.981593,37.618066,354.560838,MULTIPOLYGON (((198376.20752123397 557356.7759...,0.0,0.0


In [3]:
building_csv.sum_area.unique()

array([0.00000000e+00, 1.89583135e+03, 6.41174061e+02, ...,
       3.71621724e+03, 3.31305500e+03, 7.81312320e-01])

In [4]:
# ── 격자별 고도 + 건물 통계 처리 ─────────────────────────────────────────
import pandas as pd
import numpy as np
import os

BLDG_SRC = "/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/격자별 평균 고도 + 건물 높이 합, 건물 면적 합/grid_with_building_stats.csv"
SAVE_DIR  = "/home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/격자 기본"

BLDG_COLS = ['elev_mean', 'sum_area', 'sum_height']

# ── 로드 ─────────────────────────────────────────────────────────────────
bldg       = pd.read_csv(BLDG_SRC)
grid_basic = pd.read_csv(os.path.join(SAVE_DIR, '격자_250m_4326.csv'))
print(f'grid_basic: {grid_basic.shape}  |  bldg: {bldg.shape}')

# ── grid_basic 순서 기준 병합 (CELL_ID key) ──────────────────────────────
merged = grid_basic[['CELL_ID']].merge(
    bldg[['CELL_ID'] + BLDG_COLS], on='CELL_ID', how='left'
)
assert merged[BLDG_COLS].isna().sum().sum() == 0, '매칭 실패한 격자 있음!'
print(f'병합 완료: {merged.shape}  (결측치 없음)')

# ── [G, 3] 배열로 저장 ────────────────────────────────────────────────────
bldg_arr = merged[BLDG_COLS].values.astype(np.float32)   # [10125, 3]
print(f'\nbldg_arr shape: {bldg_arr.shape}')
for i, col in enumerate(BLDG_COLS):
    print(f'  {col:12s}: {bldg_arr[:, i].min():.2f} ~ {bldg_arr[:, i].max():.2f}')

np.save(os.path.join(SAVE_DIR, 'building_stats_static.npy'), bldg_arr)
np.save(os.path.join(SAVE_DIR, 'building_stats_cols.npy'),
        np.array(BLDG_COLS))
print(f'\n저장 완료 → {SAVE_DIR}/')
print(f'  building_stats_static.npy  {bldg_arr.shape}  (elev_mean, sum_area, sum_height)')


grid_basic: (10125, 7)  |  bldg: (10125, 11)
병합 완료: (10125, 4)  (결측치 없음)

bldg_arr shape: (10125, 3)
  elev_mean   : 2.72 ~ 688.12
  sum_area    : 0.00 ~ 39474.41
  sum_height  : 0.00 ~ 13363.21

저장 완료 → /home/data/youngwoong/ST-GNN_Dataset/Data_Preprocessed/Land Use Regression/격자 기본/
  building_stats_static.npy  (10125, 3)  (elev_mean, sum_area, sum_height)
